# 视网膜眼底图像的渗出液分割 — 运行入口（薄壳）

**代码与笔记分离**：全部实现都在仓库的 `exudate/` Python 包里；本 notebook 只负责说明与调用。

三种方法：**逻辑回归 (LR) · 支持向量机 SVM(RBF) · U-Net（预训练编码器）**。

> 运行前：`代码执行程序 → 更改运行时类型 → GPU (A100/L4)`，然后 `全部运行`，或逐格运行查看每一步的输出与图。


## 1. 环境与依赖

In [ ]:
import torch, subprocess, sys
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "无（请改用GPU运行时！）")
subprocess.run([sys.executable,"-m","pip","install","-q",
                "segmentation-models-pytorch==0.3.4","opencv-python-headless",
                "scikit-image","scikit-learn","scipy"], check=True)
print("依赖安装完成")

## 2. 克隆仓库（同时取得数据与 `exudate` 代码包）

数据在 `topic/眼底图像的渗出液分割/EX数据`，代码包在同目录下的 `topic/眼底图像的渗出液分割/exudate/`。
> 请确保已把 `exudate/` 文件夹 push 到你的仓库（与 EX数据 同级）。

In [ ]:
import os, sys, subprocess
REPO_URL = "https://github.com/LeafTraces/ML-Proj.git"          # ← 如有改动改这里
BASE = "topic/眼底图像的渗出液分割"
if not os.path.isdir("ML-Proj"):
    subprocess.run(["git","clone","--depth","1",REPO_URL], check=True)
DATA_ROOT = f"ML-Proj/{BASE}/EX数据"
PKG_DIR   = f"ML-Proj/{BASE}"                                   # exudate 包所在目录
assert os.path.isdir(os.path.join(DATA_ROOT,"images")), "未找到数据，检查 REPO_URL/BASE"
assert os.path.isdir(os.path.join(PKG_DIR,"exudate")), "未找到 exudate 包，请先把它 push 到仓库"
sys.path.insert(0, PKG_DIR)
import exudate as ex
print("已导入 exudate；数据目录:", DATA_ROOT)

## 3.（可选）调整超参

不改则用默认（672×448，U-Net=ResNet34，60 轮）。下面这格按需取消注释。

In [ ]:
# ex.configure(EPOCHS=80, BATCH=4, UNET_ENCODER="efficientnet-b3", SVM_MAX=20000)
pass

## 4. 分步运行（推荐）

四行调用即可；每一步都会打印指标、并在 `report` 时内联显示全部图表。

In [ ]:
data = ex.load_data(DATA_ROOT)        # 加载 + 预处理（出前景占比统计）

In [ ]:
clf  = ex.run_classical(data)         # 训练 逻辑回归 + RBF-SVM

In [ ]:
unet = ex.run_unet(data)             # 训练 U-Net（GPU）

In [ ]:
df = ex.report(data, clf, unet, out_dir="outputs")   # 出全部图表 + 指标表
df.set_index("Method").T

## 4b. 消融实验：ImageNet 预训练 vs 随机初始化

复用上方同一份 `data`（同一数据划分、同一随机种子），**仅改变编码器初始化方式**，
从而定量评估 ImageNet 预训练带来的增益。约需 2× 单次 U-Net 训练时长。

> 前提：cell 2 已安装 `segmentation-models-pytorch`（否则两次都会退回内置随机 U-Net，对比失效）。

In [ ]:
# 复用同一份 data（同一划分、同一种子），只改编码器初始化方式
abl = ex.run_ablation(data)          # 跑两次 U-Net：ImageNet 预训练 vs 随机初始化
abl["df"]                            # 对比表；同时已保存 ablation_table.csv / ablation.json / fig7_ablation.png

## 5. 打包下载结果（图表 + 指标，供论文/PPT 使用）

In [ ]:
ex.make_zip("outputs")
try:
    from google.colab import files; files.download("exudate_outputs.zip")
except Exception:
    print("如未自动下载，可在左侧文件面板手动下载 exudate_outputs.zip")

---
### 备选：一行跑完
```python
df = ex.run_all(DATA_ROOT)      # 等价于 load_data → run_classical → run_unet → report
```

### 把结果填进论文 / PPT
- `outputs/results_table.csv` → 论文表1 / PPT 第9页的三行（LR / SVM / U-Net）。
- `outputs/figures/fig2~fig6` → 论文图4–图7、PPT 结果页对应占位。
- `outputs/results.json` 的 `confusion` 字典 → 论文三个混淆矩阵的 TP/FP/FN/TN。
